In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تحويل الدوائر عن بُعد باستخدام خدمة Qiskit Transpiler

> **Danger:** اعتبارًا من 18 يوليو 2025، تجري عملية نقل الخدمة لدعم منصة IBM Quantum&reg; الجديدة وهي غير متاحة حاليًا. للتمريرات المعتمدة على الذكاء الاصطناعي، يمكنك استخدام [الوضع المحلي](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     الخدمة في إصدار تجريبي (beta) وقابلة للتغيير.
>     إذا كان لديك ملاحظات أو تريد التواصل مع فريق التطوير، استخدم قناة [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

توفر خدمة Qiskit Transpiler إمكانيات تحويل الدوائر الكمومية (transpilation) على السحابة. بالإضافة إلى إمكانيات محوّل Qiskit المحلي، يمكن لمهام التحويل الخاصة بك الاستفادة من موارد IBM Quantum السحابية وتمريرات المحوّل المدعومة بالذكاء الاصطناعي.

تقدم خدمة Qiskit Transpiler مكتبة Python لدمج هذه الخدمة وإمكاناتها بسلاسة في أنماط وسير عمل Qiskit الحالية. هذه الخدمة متاحة فقط لمستخدمي خطة IBM Quantum Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API).

<span id="install-transpiler-service"></span>
## تثبيت حزمة qiskit-ibm-transpiler
لاستخدام خدمة Qiskit Transpiler، ثبّت حزمة `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

تتم المصادقة تلقائيًا باستخدام [بيانات اعتماد IBM Quantum Platform](/guides/cloud-setup) الخاصة بك، بنفس الطريقة التي [يديرها Qiskit Runtime](/guides/initialize-account):
- متغير البيئة: `QISKIT_IBM_TOKEN`
- ملف الإعدادات `~/.qiskit/qiskit-ibm.json` (تحت القسم `default-ibm-quantum`).

*ملاحظة*: تتطلب هذه الحزمة Qiskit SDK الإصدار v1.X.

## خيارات transpile في qiskit-ibm-transpiler
- `backend_name` (اختياري، str) - اسم الجهاز الخلفي (backend) كما هو متوقع من قِبَل QiskitRuntimeService (مثلًا، `ibm_torino`). إذا تم تعيين هذا الخيار، تستخدم طريقة transpile التخطيط من الجهاز المحدد لعملية التحويل. إذا تم تعيين أي خيار آخر يؤثر على هذه الإعدادات، مثل `coupling_map`، يتم تجاوز إعدادات `backend_name`.
- `coupling_map` (اختياري، List[List[int]]) - قائمة coupling map صالحة (مثلًا، [[0,1],[1,2]]). إذا تم تعيينه، تستخدم طريقة transpile هذه الخريطة لعملية التحويل. إذا تم تعريفه، فإنه يتجاوز أي قيمة محددة لـ `target`.
- `optimization_level` (int) - مستوى التحسين المحتمل الذي يُطبَّق خلال عملية التحويل. القيم الصحيحة هي [1,2,3]، حيث 1 هو أقل تحسين (والأسرع)، و3 هو أعلى تحسين (والأكثر استهلاكًا للوقت).
- `ai` ("true", "false", "auto") - ما إذا كان سيتم استخدام الإمكانيات المدعومة بالذكاء الاصطناعي خلال عملية التحويل. الإمكانيات المتاحة تشمل تمريرات `AIRouting` أو طرق التوليف الأخرى المدعومة بالذكاء الاصطناعي. إذا كانت هذه القيمة `"true"`، تطبق الخدمة تمريرات تحويل مختلفة مدعومة بالذكاء الاصطناعي وفقًا لـ `optimization_level` المطلوب. إذا كانت `"false"`، تستخدم أحدث ميزات تحويل Qiskit بدون ذكاء اصطناعي. وإذا كانت `"auto"`، تقرر الخدمة ما إذا كانت ستطبق تمريرات Qiskit الاستدلالية القياسية أو التمريرات المدعومة بالذكاء الاصطناعي بناءً على دائرتك.
- `qiskit_transpile_options` (dict) - كائن قاموس Python يمكن أن يتضمن أي خيار آخر صالح في [طريقة `transpile()` في Qiskit](defaults-and-configuration-options). إذا تضمّن `qiskit_transpile_options` الخيار `optimization_level`، يتم تجاهله لصالح `optimization_level` المحدد كمعامل إدخال. إذا تضمّن `qiskit_transpile_options` أي خيار غير معروف لطريقة `transpile()` في Qiskit، ترفع المكتبة خطأً.

لمزيد من المعلومات حول طرق `qiskit-ibm-transpiler` المتاحة، راجع [مرجع qiskit-ibm-transpiler API](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). لمعرفة المزيد عن خدمة API، راجع [توثيق Qiskit Transpiler Service REST API.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## أمثلة
توضح الأمثلة التالية كيفية تحويل الدوائر باستخدام خدمة Qiskit Transpiler بمعاملات مختلفة.

1. أنشئ دائرة واستدعِ خدمة Qiskit Transpiler لتحويلها باستخدام `ibm_torino` كـ `backend_name`، و3 كـ `optimization_level`، وبدون استخدام الذكاء الاصطناعي خلال التحويل.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*ملاحظة*: يمكنك فقط استخدام أجهزة backend_name التي لديك وصول إليها عبر حساب IBM Quantum الخاص بك. بالإضافة إلى `backend_name`، يتيح `TranspilerService` أيضًا استخدام `coupling_map` كمعامل.

2. أنشئ دائرة مماثلة وحوّلها مع طلب إمكانيات التحويل بالذكاء الاصطناعي عبر تعيين الراية `ai` إلى `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. أنشئ دائرة مماثلة وحوّلها مع السماح للخدمة بالقرار حول استخدام تمريرات التحويل المدعومة بالذكاء الاصطناعي.